## 0  — Imports & device selection

In [ ]:
# !pip install torch numpy matplotlib scipy   # uncomment if not already installed
import tempfile
import os, time, csv, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
import matplotlib
matplotlib.use('inline')          # change to 'inline' if plots should appear inline
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')


# Colab uses a ProactorEventLoop on some runtimes; patch for torch compatibility
import asyncio, sys as _sys
if _sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE.upper()}')

---
## 1  — Channel Simulation  (`channel_sim.py`)

Geometry-based spatially correlated massive MIMO channel simulator.

* **ULA** / **UPA** steering vectors (half-wavelength spacing)
* Geometry-based stochastic channel model: `L` multipath components per user
* Imperfect CSI via pilot-noise model: `Ĥ = H + E`
* `MIMODataset` + `get_dataloaders` for PyTorch training

In [ ]:
# ── Steering vectors ──────────────────────────────────────────────────

def ula_steering(theta: np.ndarray, Nt: int) -> np.ndarray:
    """ULA steering vector. theta shape (...,); returns (..., Nt)."""
    n = np.arange(Nt)
    phase = np.pi * np.sin(theta)[..., np.newaxis] * n
    return (1.0 / np.sqrt(Nt)) * np.exp(1j * phase)


def upa_steering(theta: np.ndarray, phi: np.ndarray, Nv: int, Nh: int) -> np.ndarray:
    """UPA steering vector. Returns (..., Nv*Nh)."""
    av = ula_steering(theta, Nv)
    ah = ula_steering(phi,   Nh)
    a  = np.einsum('...i,...j->...ij', av, ah)
    return a.reshape(*a.shape[:-2], Nv * Nh)

In [ ]:
# ── Channel model ─────────────────────────────────────────────────────

class MassiveMIMOChannel:
    """
    Geometry-based stochastic channel model.
    Nt: TX antennas, K: users, L: multipath per user,
    array_type: 'ULA' or 'UPA', ang_spread: degrees.
    """
    def __init__(self, Nt=32, K=4, L=5, array_type='ULA',
                 Nv=4, Nh=8, ang_spread=15.0):
        assert array_type in ('ULA', 'UPA')
        if array_type == 'UPA':
            assert Nv * Nh == Nt
        self.Nt, self.K, self.L = Nt, K, L
        self.array_type = array_type
        self.Nv, self.Nh = Nv, Nh
        self.ang_spread = np.deg2rad(ang_spread)

    def generate(self, batch_size: int, rng=None):
        """Return complex channel H of shape (batch_size, K, Nt)."""
        if rng is None:
            rng = np.random.default_rng()
        Nt, K, L, B = self.Nt, self.K, self.L, batch_size

        theta_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
        delta = rng.uniform(-self.ang_spread / 2, self.ang_spread / 2, size=(B, K, L))
        theta = theta_mean[..., np.newaxis] + delta

        alpha = (rng.standard_normal((B, K, L)) +
                 1j * rng.standard_normal((B, K, L))) / np.sqrt(2)

        if self.array_type == 'ULA':
            a = ula_steering(theta, Nt)
        else:
            phi_mean = rng.uniform(-np.pi / 3, np.pi / 3, size=(B, K))
            phi = phi_mean[..., np.newaxis] + delta
            a   = upa_steering(theta, phi, self.Nv, self.Nh)

        H = np.einsum('bkl,bkln->bkn', alpha, a) / np.sqrt(L)
        H = H / np.sqrt(np.mean(np.abs(H)**2, axis=-1, keepdims=True) * Nt)
        return H

In [ ]:
# ── Imperfect CSI ─────────────────────────────────────────────────────

def add_csi_noise(H: np.ndarray, pilot_snr_db: float, rng=None) -> np.ndarray:
    """Ĥ = H + E,  E ~ CN(0, sigma_e^2 I),  sigma_e^2 = 1/(1+SNR_p)."""
    if rng is None:
        rng = np.random.default_rng()
    pilot_snr_lin = 10 ** (pilot_snr_db / 10)
    sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))
    noise = sigma_e * (rng.standard_normal(H.shape) +
                       1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    return H + noise

In [ ]:
# ── Dataset & DataLoaders ─────────────────────────────────────────────

import hashlib, json as _json


def _dataset_config_hash(cfg: dict) -> str:
    s = _json.dumps(cfg, sort_keys=True, default=str)
    return hashlib.md5(s.encode()).hexdigest()[:10]


class MIMODataset(Dataset):
    """
    Pre-generated dataset of (H_hat, H, snr_db, W_wmmse) tuples.
    Channels stored as concatenated real/imag float32 vectors.

    W_wmmse is the offline WMMSE pre-training target: the WMMSE precoder
    computed from H_hat (NOT the perfect channel — the model never sees
    perfect CSI, so its supervised target shouldn't either), flattened with
    the same convention BeamNet uses internally (real block then imag
    block, both in Nt-major / K-minor order).
    """
    def __init__(self, H_hat: np.ndarray, H: np.ndarray, snr_db: np.ndarray,
                 W_wmmse: np.ndarray):
        super().__init__()
        B, K, Nt = H.shape
        def to_real(X):
            return np.concatenate([X.real, X.imag], axis=-1).reshape(B, -1).astype(np.float32)
        self.H_hat   = torch.from_numpy(to_real(H_hat))
        self.H       = torch.from_numpy(to_real(H))
        self.snr     = torch.from_numpy(snr_db.astype(np.float32))
        self.W_wmmse = torch.from_numpy(W_wmmse.astype(np.float32))
        self.K, self.Nt = K, Nt

    def __len__(self):  return len(self.snr)
    def __getitem__(self, idx):
        return self.H_hat[idx], self.H[idx], self.snr[idx], self.W_wmmse[idx]

    def save_npz(self, path):
        """Persist this (small, in-memory) dataset — used for val/test caching."""
        np.savez(path, H_hat=self.H_hat.numpy(), H=self.H.numpy(),
                 snr=self.snr.numpy(), W_wmmse=self.W_wmmse.numpy(),
                 K=self.K, Nt=self.Nt)

    @classmethod
    def load_npz(cls, path):
        d = np.load(path)
        obj = cls.__new__(cls)
        obj.H_hat   = torch.from_numpy(d['H_hat'])
        obj.H       = torch.from_numpy(d['H'])
        obj.snr     = torch.from_numpy(d['snr'])
        obj.W_wmmse = torch.from_numpy(d['W_wmmse'])
        obj.K, obj.Nt = int(d['K']), int(d['Nt'])
        return obj


class ChunkedMIMODataset(Dataset):
    """
    Generates the dataset in small chunks and stores each chunk as a
    .npz file on disk (H_hat, H, snr, W_wmmse). Only one chunk is loaded
    into RAM at a time during training, keeping peak memory flat regardless
    of total dataset size.

    If `cache_dir` is given (e.g. a folder under Google Drive) and it
    already holds a complete, matching set of chunks + manifest, generation
    (including the expensive per-sample WMMSE solve used for pre-training
    labels) is skipped entirely and the cached chunks are reused. This is
    what lets you reopen the notebook in a fresh Colab runtime and get your
    dataloaders back in seconds instead of regenerating + re-solving WMMSE
    for the whole training set. If `cache_dir` is None, a throwaway local
    temp directory is used instead (old, non-persistent behaviour).
    """
    def __init__(self, n_samples, channel, chunk_size=10000,
                 snr_range=(0, 30), pilot_snr_range=(0, 20), seed=0,
                 wmmse_n_iter=30, wmmse_tol=1e-4,
                 cache_dir=None, force_regenerate=False):
        super().__init__()
        self.n_samples  = n_samples
        self.chunk_size = chunk_size
        self.K, self.Nt = channel.K, channel.Nt
        self._persistent = cache_dir is not None

        if cache_dir is not None:
            os.makedirs(cache_dir, exist_ok=True)
            self._cache_dir = cache_dir
        else:
            self._cache_dir = tempfile.mkdtemp(prefix='mimo_ds_')

        manifest_path = os.path.join(self._cache_dir, 'manifest.json')
        expected_config = {
            'n_samples': n_samples, 'chunk_size': chunk_size,
            'K': self.K, 'Nt': self.Nt,
            'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
            'seed': seed, 'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
        }

        cached_ok = (not force_regenerate) and os.path.exists(manifest_path)
        saved_manifest = None
        if cached_ok:
            with open(manifest_path) as f:
                saved_manifest = _json.load(f)
            cached_ok = (saved_manifest.get('config') == expected_config and
                         all(os.path.exists(p) for p in saved_manifest.get('chunk_files', [])))

        if cached_ok:
            print(f'[Dataset] Reusing cached chunks at {self._cache_dir}')
            self._chunk_files = saved_manifest['chunk_files']
            self._chunk_sizes = saved_manifest['chunk_sizes']
        else:
            self._chunk_files = []
            self._chunk_sizes = []

            rng = np.random.default_rng(seed)
            n_chunks = int(np.ceil(n_samples / chunk_size))
            written  = 0

            print(f'  Writing {n_chunks} chunks to {self._cache_dir} ...', end=' ', flush=True)
            for c in range(n_chunks):
                size = min(chunk_size, n_samples - written)
                H    = channel.generate(size, rng=rng)
                snr_db    = rng.uniform(*snr_range,       size=size).astype(np.float32)
                pilot_snr = rng.uniform(*pilot_snr_range, size=size)
                pilot_snr_lin = 10 ** (pilot_snr / 10)
                sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))[:, None, None]
                noise   = sigma_e * (rng.standard_normal(H.shape) +
                                     1j * rng.standard_normal(H.shape)) / np.sqrt(2)
                H_hat = H + noise

                # Offline WMMSE pre-training targets, computed from H_hat
                # (same info the model receives), cached alongside the chunk.
                W_wmmse = wmmse_targets_batch(H_hat, snr_db,
                                               n_iter=wmmse_n_iter, tol=wmmse_tol)

                def to_real(X):
                    B = X.shape[0]
                    return np.concatenate([X.real, X.imag], axis=-1).reshape(B, -1).astype(np.float32)

                path = os.path.join(self._cache_dir, f'chunk_{c:04d}.npz')
                np.savez(path, H_hat=to_real(H_hat), H=to_real(H),
                         snr=snr_db, W_wmmse=W_wmmse)
                self._chunk_files.append(path)
                self._chunk_sizes.append(size)
                written += size
                print(f'{c+1}', end=' ', flush=True)
            print('done.')

            with open(manifest_path, 'w') as f:
                _json.dump({'config': expected_config,
                            'chunk_files': self._chunk_files,
                            'chunk_sizes': self._chunk_sizes}, f)

        # Build flat index: each entry is (chunk_idx, row_within_chunk)
        # Kept grouped by chunk for sequential access compatibility
        self._index_map = []
        for c, sz in enumerate(self._chunk_sizes):
            for i in range(sz):
                self._index_map.append((c, i))

        self._loaded_chunk_idx  = None
        self._loaded_chunk_data = None

    def _load_chunk(self, chunk_idx):
        if self._loaded_chunk_idx != chunk_idx:
            data = np.load(self._chunk_files[chunk_idx])
            self._loaded_chunk_data = (
                torch.from_numpy(data['H_hat']),
                torch.from_numpy(data['H']),
                torch.from_numpy(data['snr']),
                torch.from_numpy(data['W_wmmse']),
            )
            self._loaded_chunk_idx = chunk_idx

    def __len__(self): return self.n_samples

    def __getitem__(self, idx):
        chunk_idx, row = self._index_map[idx]
        self._load_chunk(chunk_idx)
        H_hat, H, snr, W_wmmse = self._loaded_chunk_data
        return H_hat[row], H[row], snr[row], W_wmmse[row]

    def cleanup(self):
        """
        Deletes the chunk cache — but only if it's a local throwaway temp
        dir. Drive-backed (persistent) caches are left alone so future runs
        can reuse them without regenerating + re-solving WMMSE.
        """
        if self._persistent:
            print(f'[Dataset] Cache is persistent at {self._cache_dir} — not deleting.')
            return
        import shutil
        shutil.rmtree(self._cache_dir, ignore_errors=True)


class ChunkSequentialSampler(torch.utils.data.Sampler):
    """
    Yields indices chunk-by-chunk in shuffled order within each chunk,
    but never crosses chunk boundaries mid-batch. This means the dataset
    only ever needs one chunk loaded at a time — no disk thrashing.
    """
    def __init__(self, dataset, shuffle=True, seed=0):
        self.dataset = dataset
        self.shuffle = shuffle
        self.seed    = seed
        self.epoch   = 0

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __len__(self):
        return len(self.dataset)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        chunk_sizes  = self.dataset._chunk_sizes

        # Shuffle chunk order each epoch
        chunk_order = rng.permutation(len(chunk_sizes)).tolist()

        # Compute start offset of each chunk in the flat index
        offsets = []
        offset  = 0
        for sz in chunk_sizes:
            offsets.append(offset)
            offset += sz

        indices = []
        for c in chunk_order:
            start = offsets[c]
            sz    = chunk_sizes[c]
            chunk_indices = np.arange(start, start + sz)
            if self.shuffle:
                rng.shuffle(chunk_indices)
            indices.extend(chunk_indices.tolist())
        return iter(indices)


def generate_dataset(n_samples, channel, snr_range=(0, 30),
                     pilot_snr_range=(0, 20), seed=42,
                     wmmse_n_iter=30, wmmse_tol=1e-4,
                     cache_path=None, force_regenerate=False):
    """
    Builds an in-memory MIMODataset (used for val/test).

    If cache_path is given (a single .npz file, e.g. under Google Drive)
    and it already exists, loads straight from there instead of
    regenerating the channels and re-solving WMMSE.
    """
    if cache_path is not None and os.path.exists(cache_path) and not force_regenerate:
        print(f'[Dataset] Loading cached dataset from {cache_path}')
        return MIMODataset.load_npz(cache_path)

    rng = np.random.default_rng(seed)
    H   = channel.generate(n_samples, rng=rng)
    snr_db    = rng.uniform(*snr_range,       size=n_samples)
    pilot_snr = rng.uniform(*pilot_snr_range, size=n_samples)

    pilot_snr_lin = 10 ** (pilot_snr / 10)
    sigma_e = np.sqrt(1.0 / (1.0 + pilot_snr_lin))[:, None, None]
    noise   = sigma_e * (rng.standard_normal(H.shape) +
                         1j * rng.standard_normal(H.shape)) / np.sqrt(2)
    H_hat   = H + noise
    W_wmmse = wmmse_targets_batch(H_hat, snr_db, n_iter=wmmse_n_iter, tol=wmmse_tol)
    ds = MIMODataset(H_hat, H, snr_db, W_wmmse)

    if cache_path is not None:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        ds.save_npz(cache_path)
        print(f'[Dataset] Saved to {cache_path}')
    return ds


def get_dataloaders(Nt=32, K=4, L=5, array_type='ULA',
                    n_train=50000, n_val=5000, n_test=5000,
                    batch_size=512, num_workers=0, chunk_size=10000,
                    snr_range=(0, 30), pilot_snr_range=(0, 20),
                    wmmse_n_iter=30, wmmse_tol=1e-4,
                    cache_dir=None, force_regenerate=False):
    """
    Build train / val / test DataLoaders.
    Training set uses ChunkedMIMODataset (disk-backed, low RAM).
    Val and test use the original in-memory MIMODataset (small enough).

    cache_dir : optional path (e.g. somewhere under Google Drive). When
    given, the whole dataset — training chunks, manifest, and val/test
    .npz files, including the offline WMMSE pre-training labels — is
    cached under a config-specific subfolder (hashed from Nt/K/L/dataset
    sizes/SNR ranges/WMMSE settings/etc). Re-running this function with the
    same arguments in a later Colab session reuses that cache automatically
    instead of regenerating and re-solving WMMSE from scratch. Pass
    force_regenerate=True to rebuild anyway.

    Returns (train_loader, val_loader, test_loader, channel, train_ds, train_sampler).
    """
    Nv, Nh = 4, Nt // 4
    channel = MassiveMIMOChannel(Nt=Nt, K=K, L=L, array_type=array_type,
                                  Nv=Nv, Nh=Nh, ang_spread=15.0)

    run_cache_dir = None
    if cache_dir is not None:
        cfg = {
            'Nt': Nt, 'K': K, 'L': L, 'array_type': array_type,
            'n_train': n_train, 'n_val': n_val, 'n_test': n_test,
            'snr_range': list(snr_range), 'pilot_snr_range': list(pilot_snr_range),
            'chunk_size': chunk_size,
            'wmmse_n_iter': wmmse_n_iter, 'wmmse_tol': wmmse_tol,
        }
        cfg_hash = _dataset_config_hash(cfg)
        run_cache_dir = os.path.join(cache_dir, f'cfg_{cfg_hash}')
        os.makedirs(run_cache_dir, exist_ok=True)
        print(f'[Dataset] Cache dir: {run_cache_dir}')

    train_cache = os.path.join(run_cache_dir, 'train') if run_cache_dir else None
    print(f'[Dataset] Building chunked train set ({n_train} samples, chunk_size={chunk_size}) ...')
    train_ds      = ChunkedMIMODataset(n_train, channel, chunk_size=chunk_size,
                                        snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                                        seed=0, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                                        cache_dir=train_cache, force_regenerate=force_regenerate)
    train_sampler = ChunkSequentialSampler(train_ds, shuffle=True, seed=0)

    val_cache  = os.path.join(run_cache_dir, 'val.npz')  if run_cache_dir else None
    test_cache = os.path.join(run_cache_dir, 'test.npz') if run_cache_dir else None

    print(f'[Dataset] Generating {n_val} val samples ...')
    val_ds  = generate_dataset(n_val, channel, snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                                seed=1, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                                cache_path=val_cache, force_regenerate=force_regenerate)
    print(f'[Dataset] Generating {n_test} test samples ...')
    test_ds = generate_dataset(n_test, channel, snr_range=snr_range, pilot_snr_range=pilot_snr_range,
                                seed=2, wmmse_n_iter=wmmse_n_iter, wmmse_tol=wmmse_tol,
                                cache_path=test_cache, force_regenerate=force_regenerate)

    kw = dict(num_workers=0, pin_memory=torch.cuda.is_available())
    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=train_sampler, **kw)
    val_loader   = DataLoader(val_ds,  batch_size=batch_size, shuffle=False, **kw)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False, **kw)

    return train_loader, val_loader, test_loader, channel, train_ds, train_sampler


---
## 2  — Classical Beamformers  (`beamformers.py`)

All beamformers accept `H ∈ ℂ^{K×Nt}` and return power-normalised `W ∈ ℂ^{Nt×K}`.

| Method | Idea |
|--------|------|
| **MRT** | Matched filter — max per-user SNR, ignores IUI |
| **ZF-BF** | Pseudoinverse — nulls IUI exactly (needs Nₜ ≥ K) |
| **MMSE-BF** | Regularised ZF — balances IUI suppression vs. array gain |

In [ ]:
# ── Power normalisation ───────────────────────────────────────────────

def power_normalise(W: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    norm = np.linalg.norm(W, 'fro')
    if norm < 1e-12:
        return W
    return W * np.sqrt(P_total) / norm


# ── MRT ───────────────────────────────────────────────────────────────

def mrt(H: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    """Maximum Ratio Transmission:  W = H^H."""
    return power_normalise(H.conj().T, P_total)


# ── Zero-Forcing ──────────────────────────────────────────────────────

def zf_bf(H: np.ndarray, P_total: float = 1.0) -> np.ndarray:
    """Zero-Forcing:  W = H^H (H H^H)^{-1}."""
    HHH = H @ H.conj().T
    return power_normalise(H.conj().T @ np.linalg.pinv(HHH), P_total)


# ── MMSE / Regularised ZF ─────────────────────────────────────────────

def mmse_bf(H: np.ndarray, snr_db: float, P_total: float = 1.0) -> np.ndarray:
    """MMSE-BF:  W = H^H (H H^H + (K/SNR) I)^{-1}."""
    K   = H.shape[0]
    snr = 10 ** (snr_db / 10)
    reg = K / snr
    HHH = H @ H.conj().T
    return power_normalise(H.conj().T @ np.linalg.inv(HHH + reg * np.eye(K)), P_total)


# ── WMMSE (iterative, near-optimal reference) ─────────────────────────

def wmmse_bf(H: np.ndarray, snr_db: float, P_total: float = 1.0,
             n_iter: int = 100, tol: float = 1e-6) -> np.ndarray:
    """
    Iterative WMMSE precoder (Shi, Razaviyayn, Luo & He, IEEE TSP 2011).
    Same interface as mrt/zf_bf/mmse_bf above: single sample, H is (K,Nt) complex,
    returns W (Nt,K) complex.

    Unlike mmse_bf() (a one-shot linear MMSE/RZF beamformer -- despite the name, it
    does not iterate), this alternates U -> W -> V updates to a local optimum of the
    actual weighted sum-rate objective. This is the near-optimal reference that
    learned beamforming models are typically benchmarked against, and is also used
    below to build the supervised pre-training labels.

    Specialized to the MISO case used throughout this notebook (single-antenna users),
    so U_k, W_k are scalars per user rather than matrices.
    """
    K, Nt = H.shape
    s = 10 ** (snr_db / 10)
    sigma2 = 1.0 / s

    # RZF initialisation (WMMSE's fixed point depends on init; RZF is standard)
    V = power_normalise(H.conj().T @ np.linalg.inv(H @ H.conj().T + (K / s) * np.eye(K)), P_total)

    prev_rate = -np.inf
    for _ in range(n_iter):
        HV  = H @ V                                              # (K,K): HV[k,m] = h_k^H v_m
        hvk = np.diag(HV)                                        # desired signal per user
        u   = hvk / (np.sum(np.abs(HV) ** 2, axis=1) + sigma2)   # MMSE receive combiner
        e   = 1.0 - u.conj() * hvk                                # residual MSE
        w   = 1.0 / np.real(e).clip(min=1e-10)                    # importance weight

        diag_term = sigma2 * np.sum(w * np.abs(u) ** 2)
        A   = diag_term * np.eye(Nt) + (H.conj().T * (w * np.abs(u) ** 2)) @ H
        rhs = H.conj().T * (w * u)                                # columns = w_k u_k h_k
        V   = np.linalg.solve(A + 1e-10 * np.eye(Nt), rhs)
        V   = power_normalise(V, P_total)

        rate = compute_sum_rate(H, V, snr_db)                     # defined below in this cell
        if abs(rate - prev_rate) < tol:
            break
        prev_rate = rate

    return V


# ── SINR & Sum-Rate ───────────────────────────────────────────────────

def compute_sinr(H: np.ndarray, W: np.ndarray, snr_db: float) -> np.ndarray:
    K   = H.shape[0]
    snr = 10 ** (snr_db / 10)
    HW  = H @ W
    sinr = np.zeros(K)
    for k in range(K):
        sig   = np.abs(HW[k, k]) ** 2
        intf  = np.sum(np.abs(HW[k, :]) ** 2) - sig
        sinr[k] = sig / (intf + 1.0 / snr)
    return sinr


def compute_sum_rate(H: np.ndarray, W: np.ndarray, snr_db: float) -> float:
    return float(np.sum(np.log2(1.0 + compute_sinr(H, W, snr_db))))


# ── BER (QPSK, ZF receiver) ───────────────────────────────────────────

def compute_ber(H: np.ndarray, W: np.ndarray, snr_db: float,
                n_symbols: int = 10000, rng=None) -> float:
    """Monte-Carlo BER for QPSK using a ZF receiver."""
    if rng is None:
        rng = np.random.default_rng(0)
    K, _  = H.shape
    snr   = 10 ** (snr_db / 10)
    bits  = rng.integers(0, 2, size=(K, n_symbols, 2))
    syms  = (2 * bits - 1).astype(complex)
    syms_tx = (syms[..., 0] + 1j * syms[..., 1]) / np.sqrt(2)
    HW    = H @ W
    noise_std = np.sqrt(1 / (2 * snr))
    noise = noise_std * (rng.standard_normal((K, n_symbols)) +
                         1j * rng.standard_normal((K, n_symbols)))
    y = HW @ syms_tx + noise
    try:
        y_eq = np.linalg.solve(HW, y)
    except np.linalg.LinAlgError:
        y_eq = np.linalg.lstsq(HW, y, rcond=None)[0]
    bits_hat = np.zeros_like(bits)
    bits_hat[..., 0] = (y_eq.real > 0).astype(int)
    bits_hat[..., 1] = (y_eq.imag > 0).astype(int)
    return float(np.sum(bits_hat != bits) / bits.size)


# ── WMMSE targets for supervised pre-training ─────────────────────────
# Used only to build the offline pre-training labels (see the Dataset &
# DataLoaders section). wmmse_bf() is an iterative, per-sample numpy
# routine with no cheap batched/GPU form, so labels are generated once,
# in a process pool, and then cached to disk with the rest of the dataset.

from concurrent.futures import ProcessPoolExecutor
import os as _os

def _wmmse_flat_single(args):
    H_hat_k, snr, n_iter, tol = args
    W = wmmse_bf(H_hat_k, snr, P_total=1.0, n_iter=n_iter, tol=tol)   # (Nt, K) complex
    # same flatten convention as BeamNet._mmse_from_flat / BeamNet.forward:
    # real block (Nt-major) then imag block (Nt-major)
    return np.concatenate([W.real.ravel(), W.imag.ravel()]).astype(np.float32)

def wmmse_targets_batch(H_hat_complex: np.ndarray, snr_db: np.ndarray,
                         n_iter: int = 30, tol: float = 1e-4,
                         n_workers: int = None) -> np.ndarray:
    """
    H_hat_complex : (B, K, Nt) complex ndarray
    snr_db        : (B,) float
    returns       : (B, 2*Nt*K) float32 — flattened WMMSE precoders, one per row.

    n_iter/tol default to a looser setting than the n_iter=100 used for
    evaluation plots — these are only pre-training warm-start targets, so a
    fully-converged WMMSE solution isn't necessary and would slow dataset
    generation down a lot for little benefit.
    """
    n_workers = n_workers or max(1, (_os.cpu_count() or 2) - 1)
    args = [(H_hat_complex[i], float(snr_db[i]), n_iter, tol) for i in range(len(snr_db))]
    if n_workers == 1 or len(args) < 256:
        return np.stack([_wmmse_flat_single(a) for a in args], axis=0)
    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        out = list(ex.map(_wmmse_flat_single, args, chunksize=64))
    return np.stack(out, axis=0)


# ── Batch helper ──────────────────────────────────────────────────────

def batch_classical_metrics(H_batch: np.ndarray, snr_db_batch: np.ndarray,
                             method: str = 'mmse', P_total: float = 1.0):
    """Compute sum-rate for a batch (B, K, Nt) using a classical beamformer."""
    B = H_batch.shape[0]
    rates = np.zeros(B)
    bf_map = {'mrt': mrt, 'zf': zf_bf}
    for i in range(B):
        H, snr = H_batch[i], float(snr_db_batch[i])
        if method in ('mmse', 'wmmse'):
            W = mmse_bf(H, snr, P_total) if method == 'mmse' else wmmse_bf(H, snr, P_total)
        else:
            W = bf_map[method](H, P_total)
        rates[i] = compute_sum_rate(H, W, snr)
    return rates


---
## 3  — BeamTransformer Model  (`models.py`)

**Architecture overview**

Each of the K users' channel vectors (`Nt` complex → `2·Nt` real) becomes one **token**.  
A scalar SNR conditioning token is prepended.  
Pre-LN Transformer encoder layers apply full `(K+1)`-way self-attention at every layer —  
giving the network explicit, repeated cross-user interference awareness.

```
Input : H_hat_flat (B, 2·K·Nt) + snr_db (B,)
        ↓  embed each user's channel → (B, K, d_model)
        ↓  prepend SNR token         → (B, K+1, d_model)
        ↓  Pre-LN TransformerEncoder  (n_layers × n_heads)
        ↓  project K user tokens     → (B, K, 2·Nt)
        ↓  rearrange → W_flat        (B, 2·Nt·K)
        ↓  power-normalise           ||W||_F = 1
Output: W_flat (B, 2·Nt·K)
```

In [ ]:
# ── Power normalisation layer ─────────────────────────────────────────

class PowerNorm(nn.Module):
    """W <- sqrt(P_total) * W / ||W||_F  (per sample)."""
    def __init__(self, P_total: float = 1.0):
        super().__init__()
        self.P_total = P_total

    def forward(self, x):
        norm = torch.linalg.norm(x, dim=1, keepdim=True).clamp(min=1e-8)
        return x * (self.P_total ** 0.5) / norm


# ── BeamTransformer ───────────────────────────────────────────────────

class BeamNet(nn.Module):
    """
    Channel-token Transformer beamformer.
    Named 'BeamNet' for drop-in compatibility with checkpoints.

    Parameters
    ----------
    Nt, K    : system dimensions
    d_model  : Transformer embedding dimension
    n_layers : encoder depth
    n_heads  : attention heads (must divide d_model)
    dropout  : attention/FFN dropout
    """
    def __init__(self, Nt=32, K=4, d_model=256, n_layers=6,
                 n_heads=4, dropout=0.1, P_total=1.0):
        super().__init__()
        self.Nt, self.K = Nt, K
        self.P_total  = P_total
        self.d_model  = d_model
        self.n_layers = n_layers
        token_in = 2 * Nt

        self.token_embed = nn.Linear(token_in, d_model)
        self.snr_embed   = nn.Linear(1, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
            norm_first=True,   # Pre-LN
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=n_layers, norm=nn.LayerNorm(d_model)
        )
        self.output_proj = nn.Linear(d_model, token_in)
        self.power_norm  = PowerNorm(P_total)

    def forward(self, H_hat_flat: torch.Tensor, snr_db: torch.Tensor) -> torch.Tensor:
        B, K, Nt = H_hat_flat.shape[0], self.K, self.Nt
        half = K * Nt

        H_re = H_hat_flat[:, :half].view(B, K, Nt)
        H_im = H_hat_flat[:, half:].view(B, K, Nt)
        ch_tokens = torch.cat([H_re, H_im], dim=-1)         # (B, K, 2*Nt)

        x       = self.token_embed(ch_tokens)                # (B, K, d_model)
        snr_tok = self.snr_embed((snr_db / 30.0).unsqueeze(1)).unsqueeze(1)  # (B,1,d)
        tokens  = torch.cat([snr_tok, x], dim=1)             # (B, K+1, d_model)

        out      = self.transformer(tokens)                  # (B, K+1, d_model)
        user_out = out[:, 1:, :]                             # (B, K, d_model)
        w_tok    = self.output_proj(user_out)                # (B, K, 2*Nt)

        W_re   = w_tok[:, :, :Nt].permute(0, 2, 1).reshape(B, -1)
        W_im   = w_tok[:, :, Nt:].permute(0, 2, 1).reshape(B, -1)
        W_flat = torch.cat([W_re, W_im], dim=1)              # (B, 2*Nt*K)
        return self.power_norm(W_flat)

    def _mmse_from_flat(self, H_hat_flat: torch.Tensor,
                        snr_db: torch.Tensor) -> torch.Tensor:
        """Batch MMSE-BF — used as MSE pre-training target."""
        B, Nt, K = H_hat_flat.shape[0], self.Nt, self.K
        half = K * Nt
        H = torch.complex(H_hat_flat[:, :half].view(B, K, Nt),
                          H_hat_flat[:, half:].view(B, K, Nt))
        HHH     = H @ H.conj().transpose(-2, -1)
        snr_lin = 10.0 ** (snr_db / 10.0)
        reg     = K / snr_lin
        eye     = torch.eye(K, dtype=H.dtype, device=H.device).unsqueeze(0)
        HHH_reg = HHH + reg.view(B, 1, 1) * eye
        try:
            W = H.conj().transpose(-2, -1) @ torch.linalg.inv(HHH_reg)
        except Exception:
            W = H.conj().transpose(-2, -1) @ torch.linalg.pinv(HHH_reg)
        norm = torch.linalg.norm(W.reshape(B, -1), dim=1, keepdim=True).clamp(min=1e-8).unsqueeze(-1)
        W = W * (self.P_total ** 0.5) / norm
        return torch.cat([W.real.reshape(B, -1), W.imag.reshape(B, -1)], dim=1)


# ── Differentiable sum-rate loss ──────────────────────────────────────

def sinr_sumrate_loss(W_flat, H_flat, snr_db, Nt, K):
    """Negative sum-rate loss (differentiable through SINR)."""
    B = W_flat.shape[0]
    W = torch.complex(W_flat[:, :Nt*K].view(B, Nt, K),
                      W_flat[:, Nt*K:].view(B, Nt, K))
    H = torch.complex(H_flat[:, :K*Nt].view(B, K, Nt),
                      H_flat[:, K*Nt:].view(B, K, Nt))
    HW        = H @ W
    snr_lin   = 10.0 ** (snr_db / 10.0)
    noise_var = 1.0 / snr_lin
    sig   = torch.abs(torch.diagonal(HW, dim1=1, dim2=2)) ** 2
    total = (torch.abs(HW) ** 2).sum(dim=-1)
    sinr  = sig / (total - sig + noise_var.unsqueeze(1) + 1e-10)
    return -torch.log2(1.0 + sinr).sum(dim=1).mean()


# ── Model summary ─────────────────────────────────────────────────────

def model_summary(model, Nt, K):
    total  = sum(p.numel() for p in model.parameters())
    train_ = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print('-' * 54)
    print(f'  BeamTransformer  Nt={Nt}, K={K}  '
          f'd_model={model.d_model}  layers={model.n_layers}')
    print(f'  Total parameters : {total:,}')
    print(f'  Trainable params : {train_:,}')
    print('-' * 54)

---
## 4  — Training Infrastructure  (`train.py`)

**Training strategy**

1. **MSE pre-training** (20 epochs) — teaches BeamNet to reproduce WMMSE output
   (precomputed offline from `H_hat` and cached alongside the dataset — see
   Section 3), giving a stable, near-optimal initialisation before sum-rate
   fine-tuning.
2. **Sum-rate training** — end-to-end optimisation with the differentiable SINR layer.
3. AdamW + ReduceLROnPlateau + gradient clipping + early stopping + checkpoint saving.

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    """Return mean sum-rate (bps/Hz) over the loader."""
    model.eval()
    total, n = 0.0, 0
    for H_hat, H, snr, _ in loader:
        H_hat, H, snr = H_hat.to(device), H.to(device), snr.to(device)
        loss = sinr_sumrate_loss(model(H_hat, snr), H, snr, model.Nt, model.K)
        total += (-loss.item()) * H_hat.shape[0]
        n     += H_hat.shape[0]
    model.train()
    return total / n


class Trainer:
    def __init__(self, model, train_loader, val_loader,
                 device='cpu', lr=1e-4, max_epochs=150, patience=25,
                 grad_clip=1.0, pretrain_epochs=20,
                 checkpoint_dir='checkpoints', results_dir='results'):

        self.model           = model.to(device)
        self.train_loader    = train_loader
        self.val_loader      = val_loader
        self.device          = device
        self.max_epochs      = max_epochs
        self.patience        = patience
        self.grad_clip       = grad_clip
        self.pretrain_epochs = pretrain_epochs
        self.checkpoint_dir  = checkpoint_dir
        self.results_dir     = results_dir

        os.makedirs(checkpoint_dir, exist_ok=True)
        os.makedirs(results_dir,    exist_ok=True)

        self.optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        self.scheduler = ReduceLROnPlateau(self.optimizer, mode='max', factor=0.5,
                                           patience=5, min_lr=1e-6)
        self.mse_loss  = nn.MSELoss()
        self.best_val_rate  = -float('inf')
        self.epochs_no_impr = 0
        self.history        = []

    def train(self):
        model_summary(self.model, self.model.Nt, self.model.K)

        if self.pretrain_epochs > 0:
            print(f'\n[Pre-training] {self.pretrain_epochs} epochs with MSE loss (WMMSE target) ...')
            self._pretrain()
            print('[Pre-training] Done.\n')

        print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Val Rate":>10}  {"LR":>10}  {"Time":>8}')
        print('-' * 58)

        for epoch in range(1, self.max_epochs + 1):
            if hasattr(self.train_loader.sampler, 'set_epoch'):
                self.train_loader.sampler.set_epoch(epoch)
            t0         = time.time()
            train_loss = self._train_epoch()
            val_rate   = evaluate_model(self.model, self.val_loader, self.device)
            self.scheduler.step(val_rate)
            lr         = self.optimizer.param_groups[0]['lr']

            print(f'{epoch:>6}  {train_loss:>12.4f}  {val_rate:>10.4f}  '
                  f'{lr:>10.2e}  {time.time()-t0:>7.1f}s')

            self.history.append({'epoch': epoch, 'train_loss': train_loss,
                                  'val_rate': val_rate, 'lr': lr})

            if val_rate > self.best_val_rate:
                self.best_val_rate  = val_rate
                self.epochs_no_impr = 0
                self._save_checkpoint(epoch, val_rate)
            else:
                self.epochs_no_impr += 1
                if self.epochs_no_impr >= self.patience:
                    print(f'\n[Early stopping] No improvement for {self.patience} epochs.')
                    break

        self._save_history()
        print(f'\n[Training done] Best val sum-rate = {self.best_val_rate:.4f} bps/Hz')
        return self.history

    def _pretrain(self):
        """MSE warm-start against the precomputed WMMSE targets shipped with
        the dataset (see MIMODataset / ChunkedMIMODataset — the label is the
        4th item each batch yields)."""
        self.model.train()
        pre_opt = optim.Adam(self.model.parameters(), lr=1e-3)
        for epoch in range(1, self.pretrain_epochs + 1):
            total_loss, n = 0.0, 0
            for H_hat, _H, snr, W_wmmse in self.train_loader:
                H_hat, snr = H_hat.to(self.device), snr.to(self.device)
                W_wmmse    = W_wmmse.to(self.device)
                pre_opt.zero_grad()
                loss = self.mse_loss(self.model(H_hat, snr), W_wmmse)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
                pre_opt.step()
                total_loss += loss.item() * H_hat.shape[0]
                n          += H_hat.shape[0]
            print(f'  Pre-train epoch {epoch:>2}/{self.pretrain_epochs}  '
                  f'MSE loss = {total_loss / n:.6f}')

    def _train_epoch(self):
        self.model.train()
        total_loss, n = 0.0, 0
        for H_hat, H, snr, _ in self.train_loader:
            H_hat, H, snr = H_hat.to(self.device), H.to(self.device), snr.to(self.device)
            self.optimizer.zero_grad()
            loss = sinr_sumrate_loss(self.model(H_hat, snr), H, snr, self.model.Nt, self.model.K)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.grad_clip)
            self.optimizer.step()
            total_loss += loss.item() * H_hat.shape[0]
            n          += H_hat.shape[0]
        return total_loss / n

    def _save_checkpoint(self, epoch, val_rate):
        path = os.path.join(self.checkpoint_dir, 'beamnet_best.pt')
        torch.save({
            'epoch': epoch, 'val_rate': val_rate,
            'Nt': self.model.Nt, 'K': self.model.K,
            'hidden_dim': self.model.d_model, 'n_layers': self.model.n_layers,
            'n_heads': self.model.transformer.layers[0].self_attn.num_heads,
            'state_dict': self.model.state_dict(),
            'optimizer': self.optimizer.state_dict(),
        }, path)

    def _save_history(self):
        path = os.path.join(self.results_dir, 'training_curve.csv')
        with open(path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_rate', 'lr'])
            writer.writeheader()
            writer.writerows(self.history)
        print(f'[Training curve] Saved to {path}')


def load_best_model(checkpoint_dir='checkpoints', device='cpu'):
    path = os.path.join(checkpoint_dir, 'beamnet_best.pt')
    ckpt = torch.load(path, map_location=device)
    model = BeamNet(Nt=ckpt['Nt'], K=ckpt['K'],
                    d_model=ckpt.get('hidden_dim', 256),
                    n_layers=ckpt.get('n_layers', 6),
                    n_heads=ckpt.get('n_heads', 4))
    model.load_state_dict(ckpt['state_dict'])
    model.to(device).eval()
    print(f'[Checkpoint] Loaded epoch {ckpt["epoch"]} | val_rate = {ckpt["val_rate"]:.4f}')
    return model


---
## 5  — Evaluation Suite  (`evaluate.py`)

Six plots are generated to `results/`:

| # | File | Description |
|---|------|-------------|
| 1 | `sumrate_vs_snr.png` | Sum-rate vs. SNR — all beamformers |
| 2 | `ber_vs_snr.png` | BER vs. SNR (QPSK, log scale) |
| 3 | `beampatterns.png` | Spatial beampatterns |
| 4 | `ablation_antenna.png` | Sum-rate vs. antenna count |
| 5 | `ablation_csi_quality.png` | Sum-rate vs. pilot SNR |
| 6 | `training_curve.png` | Training/val curve from CSV |

In [ ]:
# ── Plot style ─────────────────────────────────────────────────────────
# NOTE: RESULTS_DIR is set in the Run-folder cell (Section 6 – Config).
#       This cell just records the reference so functions can use it.
#       If you run the eval suite standalone, set RESULTS_DIR manually.

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 12,
    'legend.fontsize': 10, 'lines.linewidth': 2.2,
    'lines.markersize': 6, 'grid.alpha': 0.35,
    'axes.spines.top': False, 'axes.spines.right': False,
})

COLORS  = {'MRT': '#9b59b6', 'ZF-BF': '#e67e22', 'MMSE-BF': '#2980b9',
           'WMMSE': '#f1c40f', 'BeamNet': '#27ae60', 'Perfect CSI': '#c0392b'}
MARKERS = {'MRT': 's', 'ZF-BF': '^', 'MMSE-BF': 'D', 'WMMSE': 'v',
           'BeamNet': 'o', 'Perfect CSI': 'x'}


# ── Flat tensor helpers ───────────────────────────────────────────────

def flat_to_complex_W(W_flat, Nt, K):
    half = Nt * K
    return W_flat[:half].reshape(Nt, K) + 1j * W_flat[half:].reshape(Nt, K)

def flat_to_complex_H(H_flat, K, Nt):
    half = K * Nt
    return H_flat[:half].reshape(K, Nt) + 1j * H_flat[half:].reshape(K, Nt)


# ── Plot data saver (for MATLAB reproduction) ─────────────────────────

def save_plot_data(filename, **arrays):
    """
    Save plot data as a .npz file under RESULTS_DIR/plot_data/.
    Each keyword argument becomes a named array in the file.

    Usage:
        save_plot_data('sumrate_vs_snr', snr_dbs=snr_dbs, MRT=res['MRT'], ...)

    In MATLAB, load with:
        data = load('sumrate_vs_snr.mat');   % after converting with scipy or direct npz read
    A companion .csv is also written for easy import into any tool.
    """
    import csv as _csv
    data_dir = os.path.join(RESULTS_DIR, 'plot_data')
    os.makedirs(data_dir, exist_ok=True)

    # Save .npz
    npz_path = os.path.join(data_dir, filename + '.npz')
    np.savez(npz_path, **{k: np.asarray(v) for k, v in arrays.items()})

    # Save .csv — works when all arrays share the same length (x-y style data)
    try:
        keys  = list(arrays.keys())
        vals  = [np.asarray(v).flatten() for v in arrays.values()]
        n_rows = max(len(v) for v in vals)
        csv_path = os.path.join(data_dir, filename + '.csv')
        with open(csv_path, 'w', newline='') as f:
            writer = _csv.writer(f)
            writer.writerow(keys)
            for row in zip(*[v.tolist() + [''] * (n_rows - len(v)) for v in vals]):
                writer.writerow(row)
    except Exception:
        csv_path = None  # non-uniform shapes — npz is sufficient

    print(f"[Data] {npz_path}")
    if csv_path:
        print(f"[Data] {csv_path}")


In [ ]:
# ── 1. Sum-Rate vs. SNR ───────────────────────────────────────────────

def eval_sumrate_vs_snr(model, Nt=32, K=4, n_samples=2000,
                        snr_range=(-5, 35), n_points=17, device='cpu'):
    snr_dbs = np.linspace(*snr_range, n_points)
    channel = MassiveMIMOChannel(Nt=Nt, K=K)
    rng     = np.random.default_rng(5)
    H_true  = channel.generate(n_samples, rng=rng)
    H_hat   = add_csi_noise(H_true, pilot_snr_db=15.0, rng=rng)
    results = {k: [] for k in ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE', 'BeamNet', 'Perfect CSI']
               if k != 'BeamNet' or model is not None}
    for snr in snr_dbs:
        rates = {k: [] for k in results}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['MRT']        .append(compute_sum_rate(H, mrt(Hh),             snr))
            rates['ZF-BF']      .append(compute_sum_rate(H, zf_bf(Hh),           snr))
            rates['MMSE-BF']    .append(compute_sum_rate(H, mmse_bf(Hh, snr),    snr))
            rates['WMMSE']      .append(compute_sum_rate(H, wmmse_bf(Hh, snr),   snr))
            rates['Perfect CSI'].append(compute_sum_rate(H, mmse_bf(H,  snr),    snr))
        if model is not None:
            H_re  = torch.from_numpy(H_hat.real.reshape(n_samples, -1).astype(np.float32))
            H_im  = torch.from_numpy(H_hat.imag.reshape(n_samples, -1).astype(np.float32))
            H_in  = torch.cat([H_re, H_im], dim=1).to(device)
            snr_t = torch.full((n_samples,), snr, dtype=torch.float32).to(device)
            with torch.no_grad():
                W_out = model(H_in, snr_t).cpu().numpy()
            rates['BeamNet'] = [compute_sum_rate(H_true[i],
                                flat_to_complex_W(W_out[i], Nt, K), snr)
                                for i in range(n_samples)]
        for k in results:
            results[k].append(np.mean(rates[k]))
    return snr_dbs, results


def plot_sumrate_vs_snr(snr_dbs, results, save_path, Nt=32, K=4):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, vals in results.items():
        ax.plot(snr_dbs, vals, color=COLORS[label],
                marker=MARKERS[label], markevery=2, label=label)
    ax.set_xlabel('SNR (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
    ax.set_title(f'Sum-Rate vs. SNR  —  Nt={Nt}, K={K}')
    ax.legend(); ax.grid(True)
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')

    # ── Save plot data for MATLAB ─────────────────────────────────────
    save_plot_data('sumrate_vs_snr',
                   snr_dbs=snr_dbs,
                   **{k: np.array(v) for k, v in results.items()})


In [ ]:
# ── 2. BER vs. SNR ────────────────────────────────────────────────────

def eval_ber_vs_snr(model, Nt=32, K=4, n_channels=300,
                    snr_range=(-5, 30), n_points=15, device='cpu'):
    snr_dbs = np.linspace(*snr_range, n_points)
    channel = MassiveMIMOChannel(Nt=Nt, K=K)
    rng     = np.random.default_rng(7)
    H_true  = channel.generate(n_channels, rng=rng)
    H_hat   = add_csi_noise(H_true, pilot_snr_db=15.0, rng=rng)
    results = {k: [] for k in ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE', 'BeamNet']
               if k != 'BeamNet' or model is not None}
    for snr in snr_dbs:
        rates = {k: [] for k in results}
        for i in range(n_channels):
            H, Hh = H_true[i], H_hat[i]
            rates['MRT']    .append(compute_ber(H, mrt(Hh),          snr, rng=rng))
            rates['ZF-BF']  .append(compute_ber(H, zf_bf(Hh),        snr, rng=rng))
            rates['MMSE-BF'].append(compute_ber(H, mmse_bf(Hh, snr), snr, rng=rng))
            rates['WMMSE']  .append(compute_ber(H, wmmse_bf(Hh, snr), snr, rng=rng))
        if model is not None:
            H_re  = torch.from_numpy(H_hat.real.reshape(n_channels, -1).astype(np.float32))
            H_im  = torch.from_numpy(H_hat.imag.reshape(n_channels, -1).astype(np.float32))
            H_in  = torch.cat([H_re, H_im], dim=1).to(device)
            snr_t = torch.full((n_channels,), snr, dtype=torch.float32).to(device)
            with torch.no_grad():
                W_out = model(H_in, snr_t).cpu().numpy()
            rates['BeamNet'] = [compute_ber(H_true[i],
                                flat_to_complex_W(W_out[i], Nt, K), snr, 3000, rng)
                                for i in range(n_channels)]
        for k in results:
            results[k].append(np.mean(rates[k]))
    return snr_dbs, results


def plot_ber_vs_snr(snr_dbs, results, save_path, Nt=32, K=4):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, vals in results.items():
        ax.semilogy(snr_dbs, np.maximum(vals, 1e-5), color=COLORS[label],
                    marker=MARKERS[label], markevery=2, label=label)
    ax.set_xlabel('SNR (dB)'); ax.set_ylabel('BER')
    ax.set_title(f'BER vs. SNR  —  Nt={Nt}, K={K}  (QPSK, ZF rx)')
    ax.legend(); ax.grid(True, which='both')
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')

    # ── Save plot data for MATLAB ─────────────────────────────────────
    save_plot_data('ber_vs_snr',
                   snr_dbs=snr_dbs,
                   **{k: np.array(v) for k, v in results.items()})


In [ ]:
# ── 3. Spatial Beampatterns ───────────────────────────────────────────

def plot_beampatterns(Nt=32, K=4, save_path='results/beampatterns.png'):
    angles    = np.deg2rad(np.linspace(-60, 60, K))
    a         = ula_steering(angles, Nt).astype(complex)
    H         = a
    theta_scan = np.linspace(-np.pi/2, np.pi/2, 500)
    a_scan     = ula_steering(theta_scan, Nt)

    bfs = {'MRT': mrt(H), 'ZF-BF': zf_bf(H), 'MMSE-BF': mmse_bf(H, snr_db=20)}
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
    gain_data = {}   # store for saving
    for ax, (name, W) in zip(axes, bfs.items()):
        gains_k = []
        for k in range(K):
            beam = a_scan @ W[:, k]
            pdb  = 20 * np.log10(np.abs(beam) + 1e-12)
            pdb_norm = pdb - pdb.max()
            ax.plot(np.rad2deg(theta_scan), pdb_norm,
                    color=COLORS.get(name, 'gray'), alpha=0.75)
            gains_k.append(pdb_norm)
        gain_data[name] = np.array(gains_k)   # (K, 500)
        for ang in np.rad2deg(angles):
            ax.axvline(ang, color='red', linestyle=':', linewidth=1.0, alpha=0.6)
        ax.set_title(name, color=COLORS.get(name, 'gray'))
        ax.set_xlabel('Angle (degrees)'); ax.set_ylim(-50, 5); ax.grid(True, alpha=0.35)
    axes[0].set_ylabel('Normalised Gain (dB)')
    angle_str = ', '.join(f'{int(round(np.rad2deg(a)))}deg' for a in angles)
    fig.suptitle(f'Spatial Beampatterns  —  K={K} users at {angle_str}  |  Nt={Nt} ULA',
                 fontsize=11, y=1.02)
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')

    # ── Save plot data for MATLAB ─────────────────────────────────────
    # theta_scan_deg : scan angles (500,)
    # user_angles_deg: actual user angles (K,)
    # MRT / ZF-BF / MMSE-BF: normalised gain matrices (K, 500)
    save_plot_data('beampatterns',
                   theta_scan_deg=np.rad2deg(theta_scan),
                   user_angles_deg=np.rad2deg(angles),
                   **{name: arr for name, arr in gain_data.items()})


In [ ]:
# ── 4 & 5. Ablation studies ───────────────────────────────────────────

def eval_ablation_antenna(model, K=4, pilot_snr_db=15.0, snr_db=20.0,
                          n_samples=800, device='cpu'):
    ant_counts = [8, 16, 32, 64]
    results    = {k: [] for k in ['MRT', 'ZF-BF', 'MMSE-BF', 'BeamNet', 'Perfect CSI']
                  if k != 'BeamNet' or model is not None}
    for Nt in ant_counts:
        channel = MassiveMIMOChannel(Nt=Nt, K=K)
        rng     = np.random.default_rng(10)
        H_true  = channel.generate(n_samples, rng=rng)
        H_hat   = add_csi_noise(H_true, pilot_snr_db, rng=rng)
        rates   = {k: [] for k in results}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['MRT']        .append(compute_sum_rate(H, mrt(Hh),             snr_db))
            rates['ZF-BF']      .append(compute_sum_rate(H, zf_bf(Hh),           snr_db))
            rates['MMSE-BF']    .append(compute_sum_rate(H, mmse_bf(Hh, snr_db), snr_db))
            rates['Perfect CSI'].append(compute_sum_rate(H, mmse_bf(H,  snr_db), snr_db))

        if model is not None and model.Nt == Nt:
            H_re  = torch.from_numpy(H_hat.real.reshape(n_samples, -1).astype(np.float32))
            H_im  = torch.from_numpy(H_hat.imag.reshape(n_samples, -1).astype(np.float32))
            H_in  = torch.cat([H_re, H_im], dim=1).to(device)
            snr_t = torch.full((n_samples,), snr_db, dtype=torch.float32).to(device)
            with torch.no_grad():
                W_out = model(H_in, snr_t).cpu().numpy()
            rates['BeamNet'] = [compute_sum_rate(H_true[i],
                                flat_to_complex_W(W_out[i], Nt, K), snr_db)
                                for i in range(n_samples)]
        elif 'BeamNet' in rates:
            rates['BeamNet'] = [np.nan]   # model only runs at the Nt it was trained for

        for k in results:
            results[k].append(np.mean(rates[k]))
    return ant_counts, results


def plot_ablation_antenna(ant_counts, results, save_path, K=4):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    x, width = np.arange(len(ant_counts)), 0.15
    for i, label in enumerate(['MRT', 'ZF-BF', 'MMSE-BF', 'BeamNet', 'Perfect CSI']):
        if label not in results:
            continue
        ax.bar(x + i * width, results[label], width, label=label,
               color=COLORS[label], alpha=0.85, edgecolor='white')
    ax.set_xticks(x + 2 * width)
    ax.set_xticklabels([f'Nₜ={n}' for n in ant_counts])
    ax.set_ylabel('Sum-Rate (bps/Hz)')
    ax.set_title(f'Ablation: Antenna Count  —  K={K}, SNR=20 dB')
    ax.legend(loc='upper left', ncol=2); ax.grid(axis='y')
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')



    # ── Save plot data for MATLAB ─────────────────────────────────────
    save_plot_data('ablation_antenna',
                   antenna_counts=np.array(ant_counts),
                   **{k: np.array(v) for k, v in results.items()})


def eval_ablation_csi(model, Nt=32, K=4, snr_db=20.0,
                      n_samples=800, device='cpu'):
    pilot_snrs = [0, 5, 10, 15, 20]
    results    = {k: [] for k in ['ZF-BF', 'MMSE-BF', 'BeamNet']
                  if k != 'BeamNet' or model is not None}
    channel    = MassiveMIMOChannel(Nt=Nt, K=K)
    rng        = np.random.default_rng(20)
    H_true     = channel.generate(n_samples, rng=rng)
    for p_snr in pilot_snrs:
        H_hat = add_csi_noise(H_true, p_snr, rng=rng)
        rates = {k: [] for k in results}
        for i in range(n_samples):
            H, Hh = H_true[i], H_hat[i]
            rates['ZF-BF']  .append(compute_sum_rate(H, zf_bf(Hh),           snr_db))
            rates['MMSE-BF'].append(compute_sum_rate(H, mmse_bf(Hh, snr_db), snr_db))
        if model is not None:
            H_re  = torch.from_numpy(H_hat.real.reshape(n_samples, -1).astype(np.float32))
            H_im  = torch.from_numpy(H_hat.imag.reshape(n_samples, -1).astype(np.float32))
            H_in  = torch.cat([H_re, H_im], dim=1).to(device)
            snr_t = torch.full((n_samples,), snr_db, dtype=torch.float32).to(device)
            with torch.no_grad():
                W_out = model(H_in, snr_t).cpu().numpy()
            rates['BeamNet'] = [compute_sum_rate(H_true[i],
                                flat_to_complex_W(W_out[i], Nt, K), snr_db)
                                for i in range(n_samples)]
        for k in results:
            results[k].append(np.mean(rates[k]))
    return pilot_snrs, results


def plot_ablation_csi(pilot_snrs, results, save_path, Nt=32, K=4):
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for label, vals in results.items():
        ax.plot(pilot_snrs, vals, '-o', color=COLORS[label],
                marker=MARKERS[label], label=label)
    ax.set_xlabel('Pilot SNR  ρₚ (dB)'); ax.set_ylabel('Sum-Rate (bps/Hz)')
    ax.set_title(f'Ablation: CSI Quality  —  Nt={Nt}, K={K}, Data SNR=20 dB')
    ax.legend(); ax.grid(True)
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')
    # ── Save plot data for MATLAB ─────────────────────────────────────
    save_plot_data('ablation_csi_quality',
                   pilot_snr_dbs=np.array(pilot_snrs),
                   **{k: np.array(v) for k, v in results.items()})

In [ ]:
# ── 6. Training curve ─────────────────────────────────────────────────

def plot_training_curve(csv_path='results/training_curve.csv',
                        save_path='results/training_curve.png'):
    if not os.path.exists(csv_path):
        print(f'[Skip] {csv_path} not found.'); return
    epochs, train_loss, val_rate = [], [], []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            epochs.append(int(row['epoch']))
            train_loss.append(float(row['train_loss']))
            val_rate.append(float(row['val_rate']))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(epochs, [-l for l in train_loss], color='#2980b9', label='Train Sum-Rate')
    ax1.plot(epochs, val_rate,                 color='#27ae60', label='Val Sum-Rate')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Sum-Rate (bps/Hz)')
    ax1.set_title('Training Curve'); ax1.legend(); ax1.grid(True)
    ax2.plot(epochs, train_loss, color='#e74c3c')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss (neg. sum-rate)')
    ax2.set_title('Training Loss'); ax2.grid(True)
    fig.tight_layout(); fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close(fig); print(f'[Plot] {save_path}')

    # ── Save plot data for MATLAB ─────────────────────────────────────
    save_plot_data('training_curve',
                   epoch=np.array(epochs),
                   train_loss=np.array(train_loss),
                   val_rate=np.array(val_rate))


# ── Master evaluation runner ──────────────────────────────────────────

def run_all(model=None, device='cpu', Nt=32, K=4, fast=False):
    ns = 400 if fast else 2000
    nc = 100 if fast else 300
    print('\n' + '='*55)
    print('  Evaluation Suite - Beamforming + MIMO')
    print('='*55)

    print('\n[1/6] Sum-Rate vs. SNR')
    snr_dbs, sr_res = eval_sumrate_vs_snr(model, Nt=Nt, K=K, n_samples=ns, device=device)
    plot_sumrate_vs_snr(snr_dbs, sr_res, f'{RESULTS_DIR}/sumrate_vs_snr.png', Nt=Nt, K=K)

    print('\n[2/6] BER vs. SNR')
    snr_dbs_ber, ber_res = eval_ber_vs_snr(model, Nt=Nt, K=K, n_channels=nc, device=device)
    plot_ber_vs_snr(snr_dbs_ber, ber_res, f'{RESULTS_DIR}/ber_vs_snr.png', Nt=Nt, K=K)

    print('\n[3/6] Beampatterns')
    plot_beampatterns(Nt=Nt, K=K, save_path=f'{RESULTS_DIR}/beampatterns.png')

    print('\n[4/6] Ablation - Antenna Count')
    ant_counts, ant_res = eval_ablation_antenna(
        model, K=K, n_samples=400 if fast else 800, device=device)
    plot_ablation_antenna(ant_counts, ant_res, f'{RESULTS_DIR}/ablation_antenna.png', K=K)

    print('\n[5/6] Ablation - CSI Quality')
    p_snrs, csi_res = eval_ablation_csi(
        model, Nt=Nt, K=K, n_samples=400 if fast else 800, device=device)
    plot_ablation_csi(p_snrs, csi_res, f'{RESULTS_DIR}/ablation_csi_quality.png', Nt=Nt, K=K)

    print('\n[6/6] Training Curve')
    plot_training_curve(
    csv_path=f'{RESULTS_DIR}/training_curve.csv',
    save_path=f'{RESULTS_DIR}/training_curve.png',
)

    print('\n[DONE] All plots saved to results/')

---
## 6  — Configuration & Quick Run

Edit the parameters below then run the two cells to:
1. **Train** BeamTransformer (or skip with `NO_TRAIN = True`)
2. **Evaluate** — produces all six result plots in `results/`

> Set `FAST = True` for a 2-minute demo (few samples, 5 epochs).

In [ ]:
# Mount Google Drive to persist checkpoints, results, AND the generated
# dataset (including the offline WMMSE pre-training labels) across sessions
from google.colab import drive
drive.mount("/content/drive")
import os, re, datetime

BASE_DIR = "/content/drive/MyDrive/BeamTransformer"

# ── Auto-create Run_{N} folder ────────────────────────────────────────────────
def _next_run_id(base):
    """Find the next unused Run_N index under base/."""
    existing = [d for d in os.listdir(base)
                if os.path.isdir(os.path.join(base, d)) and re.match(r'^Run_\d+$', d)] \
               if os.path.isdir(base) else []
    ids = [int(re.search(r'\d+', d).group()) for d in existing]
    return max(ids, default=0) + 1

os.makedirs(BASE_DIR, exist_ok=True)
RUN_ID  = _next_run_id(BASE_DIR)
RUN_DIR = os.path.join(BASE_DIR, f"Run_{RUN_ID}")

CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
RESULTS_DIR    = os.path.join(RUN_DIR, "results")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)

# Dataset cache lives OUTSIDE the per-run folder (at BASE_DIR level, not
# RUN_DIR) and is keyed by a hash of its own config (Nt/K/L/dataset
# sizes/SNR ranges/WMMSE settings — see get_dataloaders). That way every
# run with the same system settings reuses the same cached dataset
# (chunks + WMMSE labels) instead of regenerating it per run.
DATASET_CACHE_DIR = os.path.join(BASE_DIR, "dataset_cache")
os.makedirs(DATASET_CACHE_DIR, exist_ok=True)

print(f"Drive mounted.")
print(f"Run folder     : {RUN_DIR}")
print(f"Checkpoints    : {CHECKPOINT_DIR}")
print(f"Results        : {RESULTS_DIR}")
print(f"Dataset cache  : {DATASET_CACHE_DIR}")


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────

# System
Nt = 32          # transmit antennas
K  = 8           # single-antenna users
L  = 5           # multipath paths per user
ARRAY = 'ULA'    # 'ULA' or 'UPA'

# Dataset sizes
N_TRAIN    = 200_000
N_VAL      = 10_000
N_TEST     = 20_000
BATCH_SIZE = 512

# WMMSE pre-training labels (see wmmse_targets_batch / get_dataloaders)
WMMSE_N_ITER = 30      # iterations used only for offline pretrain-label generation
WMMSE_TOL    = 1e-4    # (eval-time WMMSE in the plots still uses the tighter n_iter=100/tol=1e-6 default)

# Dataset caching (Google Drive — see the Drive-mount cell for DATASET_CACHE_DIR)
USE_DATASET_CACHE  = True   # reuse a cached dataset (incl. WMMSE labels) if one matches
FORCE_REGENERATE_DATA = False  # True -> ignore any existing cache and rebuild

# Training
LR              = 1e-4
MAX_EPOCHS      = 100
PATIENCE        = 25
PRETRAIN_EPOCHS = 20

# Model
D_MODEL  = 128   # Transformer embedding dimension
N_LAYERS = 5   # encoder depth
N_HEADS  = 4     # attention heads

# Flags
FAST     = False  # True → tiny dataset + 5 epochs (quick demo)
NO_TRAIN = False  # True → skip training, load checkpoint

# ── Apply fast-mode overrides ─────────────────────────────────────────
if FAST:
    print('[Fast mode] Reducing dataset and epochs for quick demo.')
    N_TRAIN, N_VAL, N_TEST = 2000, 400, 400
    MAX_EPOCHS = 5;  PATIENCE = 5;  BATCH_SIZE = 128
    PRETRAIN_EPOCHS = 3
    WMMSE_N_ITER = 15  # keep fast-mode dataset generation quick too

# ── Save run parameters to text file ─────────────────────────────────
def save_run_params(path):
    lines = [
        f"Run ID          : {RUN_ID}",
        f"Timestamp       : {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Device          : {DEVICE}",
        "",
        "=== System ===",
        f"Array type      : {ARRAY}",
        f"Nt (TX antennas): {Nt}",
        f"K  (users)      : {K}",
        f"L  (paths/user) : {L}",
        "",
        "=== Dataset ===",
        f"N_TRAIN         : {N_TRAIN}",
        f"N_VAL           : {N_VAL}",
        f"N_TEST          : {N_TEST}",
        f"BATCH_SIZE      : {BATCH_SIZE}",
        "",
        "=== Training ===",
        f"LR              : {LR}",
        f"MAX_EPOCHS      : {MAX_EPOCHS}",
        f"PATIENCE        : {PATIENCE}",
        f"PRETRAIN_EPOCHS : {PRETRAIN_EPOCHS}",
        f"FAST mode       : {FAST}",
        f"NO_TRAIN        : {NO_TRAIN}",
        "",
        "=== Model (BeamTransformer) ===",
        f"D_MODEL         : {D_MODEL}",
        f"N_LAYERS        : {N_LAYERS}",
        f"N_HEADS         : {N_HEADS}",
        "",
        "=== Paths ===",
        f"Run dir         : {RUN_DIR}",
        f"Checkpoints     : {CHECKPOINT_DIR}",
        f"Results         : {RESULTS_DIR}",
    ]
    with open(path, 'w') as f:
        f.write("\n".join(lines))
    print(f"[Params] Saved to {path}")

params_path = os.path.join(RUN_DIR, "run_params.txt")
save_run_params(params_path)

print(f'Device: {DEVICE}  |  Array: {ARRAY}  |  Nt={Nt}  |  K={K}  |  L={L}')


In [ ]:
# ── Build data ────────────────────────────────────────────────────────

train_loader, val_loader, test_loader, channel, train_ds, train_sampler = get_dataloaders(
    Nt=Nt, K=K, L=L, array_type=ARRAY,
    n_train=N_TRAIN, n_val=N_VAL, n_test=N_TEST,
    batch_size=BATCH_SIZE, chunk_size=10_000,
    wmmse_n_iter=WMMSE_N_ITER, wmmse_tol=WMMSE_TOL,
    cache_dir=(DATASET_CACHE_DIR if USE_DATASET_CACHE else None),
    force_regenerate=FORCE_REGENERATE_DATA,
)

# ── Build model ───────────────────────────────────────────────────────

model = BeamNet(Nt=Nt, K=K, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS)
total_params = sum(p.numel() for p in model.parameters())

if not NO_TRAIN:
    print('\n[Phase 1] Training BeamTransformer\n')
    trainer = Trainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        device=DEVICE, lr=LR, max_epochs=MAX_EPOCHS, patience=PATIENCE,
        pretrain_epochs=PRETRAIN_EPOCHS,
        checkpoint_dir=CHECKPOINT_DIR, results_dir=RESULTS_DIR,
    )
    trainer.train()
    train_ds.cleanup()   # no-op if the dataset is Drive-cached (persistent); else deletes local temp chunks

# ── Load best checkpoint ──────────────────────────────────────────────

ckpt_path = os.path.join(CHECKPOINT_DIR, 'beamnet_best.pt')
if os.path.exists(ckpt_path):
    model = load_best_model(CHECKPOINT_DIR, device=DEVICE)
else:
    print('[Warning] No checkpoint found — evaluating without BeamNet.')
    model = None

# ── BeamNet val / test sum-rate ───────────────────────────────────────

val_rate_beamnet  = None
test_rate_beamnet = None

if model is not None:
    val_rate_beamnet  = evaluate_model(model, val_loader,  DEVICE)
    test_rate_beamnet = evaluate_model(model, test_loader, DEVICE)
    print(f'\n[Val  Set] BeamNet mean sum-rate = {val_rate_beamnet:.4f} bps/Hz')
    print(f'[Test Set] BeamNet mean sum-rate = {test_rate_beamnet:.4f} bps/Hz')

# ── Classical baseline sum-rates (val set, SNR grid) ─────────────────
# Compute mean sum-rate at a reference SNR=20 dB for a quick comparison.

print('\n[Baselines] Computing classical beamformer rates on val set (SNR=20 dB) ...')
_ref_snr   = 20.0
_n_samples = min(500, N_VAL)
_rng       = np.random.default_rng(99)
_H_true    = channel.generate(_n_samples, rng=_rng)
_H_hat     = add_csi_noise(_H_true, 15.0, rng=_rng)

_rates = {'MRT': [], 'ZF-BF': [], 'MMSE-BF': [], 'WMMSE': [], 'Perfect CSI': []}
for i in range(_n_samples):
    H, Hh = _H_true[i], _H_hat[i]
    _rates['MRT']        .append(compute_sum_rate(H, mrt(Hh),              _ref_snr))
    _rates['ZF-BF']      .append(compute_sum_rate(H, zf_bf(Hh),            _ref_snr))
    _rates['MMSE-BF']    .append(compute_sum_rate(H, mmse_bf(Hh, _ref_snr), _ref_snr))
    _rates['WMMSE']      .append(compute_sum_rate(H, wmmse_bf(Hh, _ref_snr), _ref_snr))
    _rates['Perfect CSI'].append(compute_sum_rate(H, mmse_bf(H,  _ref_snr), _ref_snr))

baseline_rates_20dB = {k: float(np.mean(v)) for k, v in _rates.items()}
if model is not None:
    _H_re  = torch.from_numpy(_H_hat.real.reshape(_n_samples, -1).astype(np.float32))
    _H_im  = torch.from_numpy(_H_hat.imag.reshape(_n_samples, -1).astype(np.float32))
    _H_in  = torch.cat([_H_re, _H_im], dim=1).to(DEVICE)
    _snr_t = torch.full((_n_samples,), _ref_snr, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        _W_out = model(_H_in, _snr_t).cpu().numpy()
    _bn_rates = [compute_sum_rate(_H_true[i], flat_to_complex_W(_W_out[i], Nt, K), _ref_snr)
                 for i in range(_n_samples)]
    baseline_rates_20dB['BeamNet'] = float(np.mean(_bn_rates))

print(f'  MRT        : {baseline_rates_20dB.get("MRT",        0):.4f} bps/Hz')
print(f'  ZF-BF      : {baseline_rates_20dB.get("ZF-BF",      0):.4f} bps/Hz')
print(f'  MMSE-BF    : {baseline_rates_20dB.get("MMSE-BF",    0):.4f} bps/Hz')
print(f'  WMMSE      : {baseline_rates_20dB.get("WMMSE",      0):.4f} bps/Hz')
print(f'  Perfect CSI: {baseline_rates_20dB.get("Perfect CSI",0):.4f} bps/Hz')
if 'BeamNet' in baseline_rates_20dB:
    print(f'  BeamNet    : {baseline_rates_20dB["BeamNet"]:.4f} bps/Hz')


In [ ]:
# ── Run full evaluation suite ─────────────────────────────────────────

print('\n[Phase 2] Running evaluation suite\n')
run_all(model=model, device=DEVICE, Nt=Nt, K=K, fast=FAST)

print('\n[DONE]')
print(f'  Run folder : {RUN_DIR}/')
print(f'  Plots      : {RESULTS_DIR}/')
print(f'  Plot data  : {RESULTS_DIR}/plot_data/   (npz + csv for MATLAB)')
print(f'  Model      : {CHECKPOINT_DIR}/beamnet_best.pt')
print(f'  Curve      : {RESULTS_DIR}/training_curve.csv')
print(f'  Params     : {RUN_DIR}/run_params.txt')


In [ ]:
# ── Generate run report (.docx) ───────────────────────────────────────
# Produces a Word document summarising this run: parameters, training
# curve stats, val/test rates, baseline comparison, and sample data.
# Requires: pip install python-docx  (pure Python, no Node needed)
# !pip install python-docx --break-system-packages  # uncomment in Colab

import importlib, subprocess, sys

def _ensure_python_docx():
    if importlib.util.find_spec('docx') is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                               'python-docx', '--break-system-packages', '-q'])

_ensure_python_docx()
from docx import Document as _Document
from docx.shared import Pt, RGBColor, Inches, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
import csv as _csv_mod

def _set_cell_bg(cell, hex_color):
    """Set table cell background shading."""
    tc   = cell._tc
    tcPr = tc.get_or_add_tcPr()
    shd  = OxmlElement('w:shd')
    shd.set(qn('w:val'),   'clear')
    shd.set(qn('w:color'), 'auto')
    shd.set(qn('w:fill'),  hex_color)
    tcPr.append(shd)

def _hdr_row(table, texts, bg='1F3864', fg='FFFFFF'):
    """Format a header row: dark navy bg, white bold text."""
    row = table.rows[0]
    for cell, text in zip(row.cells, texts):
        cell.text = ''
        run = cell.paragraphs[0].add_run(text)
        run.bold = True
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        run.font.size = Pt(9)
        _set_cell_bg(cell, bg)

def _add_row(table, values, bold_first=False, alt=False):
    """Add a data row, optionally with alternating shading."""
    row = table.add_row()
    for j, val in enumerate(values):
        cell = row.cells[j]
        cell.text = str(val)
        p = cell.paragraphs[0]
        p.runs[0].font.size = Pt(9)
        if bold_first and j == 0:
            p.runs[0].bold = True
        if alt:
            _set_cell_bg(cell, 'EBF3FB')

def _section_heading(doc, text, level=1):
    p = doc.add_heading(text, level=level)
    for run in p.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)
    return p

def generate_run_report(run_dir, results_dir, checkpoint_dir,
                        run_id, params_dict,
                        val_rate_beamnet, test_rate_beamnet,
                        baseline_rates_20dB,
                        model=None, channel=None, device='cpu',
                        Nt=32, K=4, n_report_samples=3):
    """
    Generate a Word (.docx) report for one training run.
    Mirrors the style of the sample Training_Data_Samples.docx.
    """
    doc = _Document()

    # ── Page margins ──────────────────────────────────────────────────
    for section in doc.sections:
        section.top_margin    = Cm(2.0)
        section.bottom_margin = Cm(2.0)
        section.left_margin   = Cm(2.5)
        section.right_margin  = Cm(2.0)

    # ── Title ─────────────────────────────────────────────────────────
    title = doc.add_heading(f'BeamTransformer — Run {run_id} Report', 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for run in title.runs:
        run.font.color.rgb = RGBColor(0x1F, 0x38, 0x64)

    # ── Meta-info table ───────────────────────────────────────────────
    meta = [
        ('Author',      params_dict.get('author', 'Tarvir Anjum Aditto')),
        ('Course',      params_dict.get('course', 'Broadband Wireless Communication')),
        ('Date',        params_dict.get('timestamp', datetime.datetime.now().strftime('%d %B, %Y'))),
        ('Device',      params_dict.get('device', device)),
        ('System',      f"Nt={params_dict.get('Nt', Nt)}, K={params_dict.get('K', K)}, "
                        f"{params_dict.get('array', 'ULA')}, L={params_dict.get('L',5)}, "
                        f"pilot SNR=15 dB"),
        ('Model',       f"BeamTransformer  d_model={params_dict.get('d_model',256)}, "
                        f"layers={params_dict.get('n_layers',6)}, "
                        f"heads={params_dict.get('n_heads',4)}  "
                        f"({sum(p.numel() for p in model.parameters()):,} params)"
                        if model is not None else 'N/A'),
    ]
    tbl = doc.add_table(rows=len(meta), cols=2)
    tbl.style = 'Table Grid'
    for i, (k, v) in enumerate(meta):
        tbl.rows[i].cells[0].text = k
        tbl.rows[i].cells[1].text = v
        tbl.rows[i].cells[0].paragraphs[0].runs[0].bold = True
        tbl.rows[i].cells[0].paragraphs[0].runs[0].font.size = Pt(10)
        tbl.rows[i].cells[1].paragraphs[0].runs[0].font.size = Pt(10)
        _set_cell_bg(tbl.rows[i].cells[0], 'D6E4F0')
    doc.add_paragraph()

    # ══════════════════════════════════════════════════════════════════
    # SECTION 1 — Training Summary
    # ══════════════════════════════════════════════════════════════════
    _section_heading(doc, '1. Training Summary')

    csv_path = os.path.join(results_dir, 'training_curve.csv')
    epochs_done, best_val, final_train_loss = 0, float('nan'), float('nan')
    if os.path.exists(csv_path):
        rows = []
        with open(csv_path) as f:
            rows = list(_csv_mod.DictReader(f))
        if rows:
            epochs_done      = int(rows[-1]['epoch'])
            best_val         = max(float(r['val_rate']) for r in rows)
            final_train_loss = float(rows[-1]['train_loss'])

    tbl2 = doc.add_table(rows=1, cols=2)
    tbl2.style = 'Table Grid'
    _hdr_row(tbl2, ['Metric', 'Value'])
    summary_rows = [
        ('Epochs completed',  str(epochs_done)),
        ('Best val sum-rate', f'{best_val:.4f} bps/Hz' if not np.isnan(best_val) else 'N/A'),
        ('Final train loss',  f'{final_train_loss:.4f}' if not np.isnan(final_train_loss) else 'N/A'),
        ('Val  set sum-rate', f'{val_rate_beamnet:.4f} bps/Hz'  if val_rate_beamnet  is not None else 'N/A'),
        ('Test set sum-rate', f'{test_rate_beamnet:.4f} bps/Hz' if test_rate_beamnet is not None else 'N/A'),
    ]
    for i, (k, v) in enumerate(summary_rows):
        _add_row(tbl2, [k, v], bold_first=True, alt=(i % 2 == 1))
    doc.add_paragraph()

    # ══════════════════════════════════════════════════════════════════
    # SECTION 2 — Baseline Comparison at SNR=20 dB
    # ══════════════════════════════════════════════════════════════════
    _section_heading(doc, '2. Baseline Comparison at SNR = 20 dB')
    doc.add_paragraph(
        'Mean sum-rate (bps/Hz) computed over 500 val-set samples. '
        'BeamNet uses the best checkpoint; classical beamformers use noisy CSI '
        'with pilot SNR = 15 dB. Perfect CSI uses the true channel.'
    ).runs[0].font.size = Pt(10)

    mmse_ref = baseline_rates_20dB.get('MMSE-BF', 1.0) or 1.0
    methods_order = ['MRT', 'ZF-BF', 'MMSE-BF', 'WMMSE', 'BeamNet', 'Perfect CSI']
    tbl3 = doc.add_table(rows=1, cols=3)
    tbl3.style = 'Table Grid'
    _hdr_row(tbl3, ['Method', 'Sum-Rate (bps/Hz)', 'Gain vs MMSE-BF'])
    for i, m in enumerate(methods_order):
        if m not in baseline_rates_20dB:
            continue
        rate = baseline_rates_20dB[m]
        gain_str = '—' if m == 'MMSE-BF' else \
                   f'{rate - mmse_ref:+.4f}  ({(rate - mmse_ref)/mmse_ref*100:+.1f}%)'
        _add_row(tbl3, [m, f'{rate:.4f}', gain_str], bold_first=True, alt=(i % 2 == 1))
    doc.add_paragraph()

    # ══════════════════════════════════════════════════════════════════
    # SECTION 3 — Sample Data (H_hat, W_mmse, W_beamnet, SINR)
    # ══════════════════════════════════════════════════════════════════
    _section_heading(doc, '3. Sample CSI & Precoding Matrices')

    snr_samples = [5.0, 15.0, 25.0][:n_report_samples]

    if channel is None or model is None:
        doc.add_paragraph('[Skipped — model or channel not available]').italic = True
    else:
        rng = np.random.default_rng(42)
        H_true_all = channel.generate(n_report_samples, rng=rng)
        H_hat_all  = add_csi_noise(H_true_all, 15.0, rng=rng)

        for s_idx, snr_db in enumerate(snr_samples):
            H_true = H_true_all[s_idx]   # (K, Nt)
            H_hat  = H_hat_all[s_idx]

            _section_heading(doc, f'Sample {s_idx+1}  —  Data SNR = {snr_db:.0f} dB', level=2)

            # 3a. H_hat magnitude (every 4th antenna)
            _section_heading(doc, f'|H_hat[k, n]|  (every 4th antenna)', level=3)
            step  = max(1, Nt // 8)
            ants  = list(range(0, Nt, step))
            tbl_h = doc.add_table(rows=1, cols=1 + len(ants))
            tbl_h.style = 'Table Grid'
            _hdr_row(tbl_h, ['User \\ Ant'] + [f'n={a}' for a in ants])
            for k in range(K):
                vals = [f'{np.abs(H_hat[k, a]):.4f}' for a in ants]
                _add_row(tbl_h, [f'user {k}'] + vals, bold_first=True, alt=(k % 2 == 1))
            doc.add_paragraph()

            # 3b. W_mmse magnitude (every 2nd antenna)
            W_mmse = mmse_bf(H_hat, snr_db)
            _section_heading(doc, f'|W_mmse[n, k]|  (every 2nd antenna)', level=3)
            step2 = max(1, Nt // 16)
            ants2 = list(range(0, Nt, step2))
            tbl_wm = doc.add_table(rows=1, cols=1 + K)
            tbl_wm.style = 'Table Grid'
            _hdr_row(tbl_wm, ['Ant \\ User'] + [f'u{k}' for k in range(K)])
            for a in ants2:
                vals = [f'{np.abs(W_mmse[a, k]):.4f}' for k in range(K)]
                _add_row(tbl_wm, [f'ant {a:>2}'] + vals, bold_first=True,
                         alt=(ants2.index(a) % 2 == 1))
            p_norm = doc.add_paragraph()
            run_n  = p_norm.add_run(f'||W_mmse||_F = {np.linalg.norm(W_mmse, "fro"):.6f}   '
                                    f'(Total power = {np.linalg.norm(W_mmse, "fro")**2:.6f})')
            run_n.bold = True
            run_n.font.size = Pt(10)
            doc.add_paragraph()

            # 3c. W_beamnet magnitude
            H_re = torch.from_numpy(H_hat.real.reshape(1, -1).astype(np.float32)).to(device)
            H_im = torch.from_numpy(H_hat.imag.reshape(1, -1).astype(np.float32)).to(device)
            H_in = torch.cat([H_re, H_im], dim=1)
            snr_t = torch.tensor([snr_db], dtype=torch.float32).to(device)
            with torch.no_grad():
                W_bn_flat = model(H_in, snr_t).cpu().numpy()[0]
            W_bn = flat_to_complex_W(W_bn_flat, Nt, K)

            _section_heading(doc, f'|W_beamnet[n, k]|  (every 2nd antenna)', level=3)
            tbl_wb = doc.add_table(rows=1, cols=1 + K)
            tbl_wb.style = 'Table Grid'
            _hdr_row(tbl_wb, ['Ant \\ User'] + [f'u{k}' for k in range(K)])
            for a in ants2:
                vals = [f'{np.abs(W_bn[a, k]):.4f}' for k in range(K)]
                _add_row(tbl_wb, [f'ant {a:>2}'] + vals, bold_first=True,
                         alt=(ants2.index(a) % 2 == 1))
            p_norm2 = doc.add_paragraph()
            run_n2  = p_norm2.add_run(f'||W_beamnet||_F = {np.linalg.norm(W_bn, "fro"):.6f}   '
                                       f'(Total power = {np.linalg.norm(W_bn, "fro")**2:.6f})')
            run_n2.bold = True
            run_n2.font.size = Pt(10)
            doc.add_paragraph()

            # 3d. Sum-rate comparison
            sr_mmse = compute_sum_rate(H_true, W_mmse, snr_db)
            sr_bn   = compute_sum_rate(H_true, W_bn,   snr_db)
            diff_sr = sr_bn - sr_mmse
            _section_heading(doc, 'Sum-Rate Comparison  (evaluated on true channel H)', level=3)
            tbl_sr = doc.add_table(rows=1, cols=3)
            tbl_sr.style = 'Table Grid'
            _hdr_row(tbl_sr, ['Method', 'Sum-Rate (bps/Hz)', 'Gain vs MMSE-BF'])
            _add_row(tbl_sr, ['MMSE-BF', f'{sr_mmse:.4f}', '—'], bold_first=True)
            _add_row(tbl_sr, ['BeamNet', f'{sr_bn:.4f}',
                               f'{diff_sr:+.4f}  ({diff_sr/max(sr_mmse,1e-6)*100:+.1f}%)'],
                     bold_first=True, alt=True)
            doc.add_paragraph()

            # 3e. Per-user SINR breakdown
            _section_heading(doc, 'Per-User SINR Breakdown', level=3)
            doc.add_paragraph(
                f'SINR_k = |h_k^H w_k|^2 / (sum_{{j≠k}} |h_k^H w_j|^2 + sigma^2). '
                f'Evaluated on true channel H at SNR = {snr_db:.0f} dB.'
            ).runs[0].font.size = Pt(9)

            sinr_mmse   = compute_sinr(H_true, W_mmse, snr_db)
            sinr_bn     = compute_sinr(H_true, W_bn,   snr_db)
            rate_mmse_k = np.log2(1.0 + sinr_mmse)
            rate_bn_k   = np.log2(1.0 + sinr_bn)

            tbl_sinr = doc.add_table(rows=1, cols=6)
            tbl_sinr.style = 'Table Grid'
            _hdr_row(tbl_sinr, ['User',
                                 'SINR MMSE-BF', 'Rate MMSE-BF (bps/Hz)',
                                 'SINR BeamNet',  'Rate BeamNet (bps/Hz)',
                                 'Rate Gain'])
            for k in range(K):
                gain_k = rate_bn_k[k] - rate_mmse_k[k]
                _add_row(tbl_sinr,
                         [f'user {k}',
                          f'{sinr_mmse[k]:.4f}', f'{rate_mmse_k[k]:.4f}',
                          f'{sinr_bn[k]:.4f}',   f'{rate_bn_k[k]:.4f}',
                          f'{gain_k:+.4f}'],
                         bold_first=True, alt=(k % 2 == 1))
            _add_row(tbl_sinr,
                     ['TOTAL', '—', f'{rate_mmse_k.sum():.4f}',
                      '—',     f'{rate_bn_k.sum():.4f}',
                      f'{(rate_bn_k - rate_mmse_k).sum():+.4f}'],
                     bold_first=True)
            doc.add_paragraph()

    # ══════════════════════════════════════════════════════════════════
    # SECTION 4 — Key Parameters
    # ══════════════════════════════════════════════════════════════════
    _section_heading(doc, '4. Full Hyperparameter Record')
    doc.add_paragraph(
        'A complete copy of all parameters is also saved as run_params.txt '
        'in the run folder.'
    ).runs[0].font.size = Pt(10)

    param_items = [
        ('Array type',      params_dict.get('array',    'ULA')),
        ('Nt',              str(params_dict.get('Nt',    32))),
        ('K',               str(params_dict.get('K',      4))),
        ('L',               str(params_dict.get('L',      5))),
        ('N_TRAIN',         str(params_dict.get('n_train', 'N/A'))),
        ('N_VAL',           str(params_dict.get('n_val',   'N/A'))),
        ('N_TEST',          str(params_dict.get('n_test',  'N/A'))),
        ('BATCH_SIZE',      str(params_dict.get('batch_size', 512))),
        ('LR',              str(params_dict.get('lr', 1e-4))),
        ('MAX_EPOCHS',      str(params_dict.get('max_epochs', 80))),
        ('PATIENCE',        str(params_dict.get('patience', 25))),
        ('PRETRAIN_EPOCHS', str(params_dict.get('pretrain_epochs', 20))),
        ('D_MODEL',         str(params_dict.get('d_model', 256))),
        ('N_LAYERS',        str(params_dict.get('n_layers', 6))),
        ('N_HEADS',         str(params_dict.get('n_heads', 4))),
    ]
    tbl4 = doc.add_table(rows=1, cols=2)
    tbl4.style = 'Table Grid'
    _hdr_row(tbl4, ['Parameter', 'Value'])
    for i, (k, v) in enumerate(param_items):
        _add_row(tbl4, [k, v], bold_first=True, alt=(i % 2 == 1))

    # ── Save ──────────────────────────────────────────────────────────
    report_path = os.path.join(run_dir, f'Run_{run_id}_Report.docx')
    doc.save(report_path)
    print(f'[Report] Saved to {report_path}')
    return report_path


# ── Build params dict from current run ───────────────────────────────
_params_dict = {
    'author':          'Tarvir Anjum Aditto',
    'course':          'Broadband Wireless Communication — MSc Project',
    'timestamp':       datetime.datetime.now().strftime('%d %B, %Y'),
    'device':          DEVICE,
    'array':           ARRAY,
    'Nt':              Nt,  'K': K,  'L': L,
    'n_train':         N_TRAIN,
    'n_val':           N_VAL,
    'n_test':          N_TEST,
    'batch_size':      BATCH_SIZE,
    'lr':              LR,
    'max_epochs':      MAX_EPOCHS,
    'patience':        PATIENCE,
    'pretrain_epochs': PRETRAIN_EPOCHS,
    'd_model':         D_MODEL,
    'n_layers':        N_LAYERS,
    'n_heads':         N_HEADS,
}

report_path = generate_run_report(
    run_dir=RUN_DIR,
    results_dir=RESULTS_DIR,
    checkpoint_dir=CHECKPOINT_DIR,
    run_id=RUN_ID,
    params_dict=_params_dict,
    val_rate_beamnet=val_rate_beamnet,
    test_rate_beamnet=test_rate_beamnet,
    baseline_rates_20dB=baseline_rates_20dB,
    model=model,
    channel=channel,
    device=DEVICE,
    Nt=Nt, K=K,
)
print(f'\n[Summary]')
print(f'  Val  sum-rate  : {val_rate_beamnet:.4f} bps/Hz'  if val_rate_beamnet  is not None else '  Val  sum-rate  : N/A')
print(f'  Test sum-rate  : {test_rate_beamnet:.4f} bps/Hz' if test_rate_beamnet is not None else '  Test sum-rate  : N/A')
print(f'  Report         : {report_path}')

---
## 7  — Display saved plots (optional)

The cells below load saved PNG files and display them inline.  
Switch `matplotlib.use('Agg')` at the top to `'inline'` if you prefer plots to render directly.

In [ ]:
from IPython.display import Image, display

plot_files = [
    f'{RESULTS_DIR}/sumrate_vs_snr.png',
    f'{RESULTS_DIR}/ber_vs_snr.png',
    f'{RESULTS_DIR}/beampatterns.png',
    f'{RESULTS_DIR}/ablation_antenna.png',
    f'{RESULTS_DIR}/ablation_csi_quality.png',
    f'{RESULTS_DIR}/training_curve.png',
]

for p in plot_files:
    if os.path.exists(p):
        print(f'\n── {p} ──')
        display(Image(filename=p))
    else:
        print(f'[Missing] {p}')
